# DQL: Set Operations

Set operations combine the results of two compatible queries. Compatible means the same number of columns and compatible data types.


## Setup

Run this first. It creates a Spark session, finds the CSV files, loads them as DataFrames, and registers SQL temp views.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd(),
]

DATA_DIR = next((path for path in candidate_dirs if (path / "emp.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find emp.csv. Put the CSV files in ./data or beside this notebook.")

print(f"Reading CSV files from: {DATA_DIR}")

emp = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp.csv"))
dept = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "dept.csv"))
salgrade = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "salgrade.csv"))
projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "projects.csv"))
emp_projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp_projects.csv"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## Prepare Example Sets

These small DataFrames make set operation behavior easy to see.


In [ ]:
sales_people = emp.where(F.col("job") == "salesman").select("empno", "ename")
high_salary_people = emp.where(F.col("sal") >= 3000).select("empno", "ename")

sales_people.show()
high_salary_people.show()


## UNION

SQL `UNION` removes duplicates. In the DataFrame API, use `unionByName(...).distinct()`.


In [ ]:
sales_people.unionByName(high_salary_people).distinct().orderBy("empno").show()

spark.sql("""
SELECT empno, ename FROM emp WHERE job = 'salesman'
UNION
SELECT empno, ename FROM emp WHERE sal >= 3000
ORDER BY empno
""").show()


## UNION ALL

`UNION ALL` keeps duplicate rows.


In [ ]:
sales_people.unionByName(high_salary_people).orderBy("empno").show()

spark.sql("""
SELECT empno, ename FROM emp WHERE job = 'salesman'
UNION ALL
SELECT empno, ename FROM emp WHERE sal >= 3000
ORDER BY empno
""").show()


## INTERSECT

`INTERSECT` returns rows found in both result sets.


In [ ]:
sales_people.intersect(high_salary_people).orderBy("empno").show()

spark.sql("""
SELECT empno, ename FROM emp WHERE job = 'salesman'
INTERSECT
SELECT empno, ename FROM emp WHERE sal >= 3000
ORDER BY empno
""").show()


## EXCEPT

`EXCEPT` returns rows from the first query that are not in the second query.


In [ ]:
sales_people.subtract(high_salary_people).orderBy("empno").show()

spark.sql("""
SELECT empno, ename FROM emp WHERE job = 'salesman'
EXCEPT
SELECT empno, ename FROM emp WHERE sal >= 3000
ORDER BY empno
""").show()
